To apply GridSearchCV to the XGBoost Classifier, I import GridSearchCV and XGBClassifier, define a hyperparameter grid, initialize the classifier, and fit GridSearchCV to my RFE-selected training data. Then, I evaluate the best model’s performance on the test set using classification metrics.

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize XGBClassifier instance with base parameters
# The objective and num_class are fixed for this multiclass problem
# random_state for reproducibility
base_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train['Attack_type'].unique()),
    eval_metric='mlogloss',
    use_label_encoder=False, # Suppress deprecation warning
    random_state=42
)

# 2. Define a parameter grid for the XGBoost Classifier
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

# 3. Initialize GridSearchCV
# cv=3 for 3-fold cross-validation
# scoring='accuracy' to optimize for accuracy
# n_jobs=-1 to use all available CPU cores for parallel processing
grid_search = GridSearchCV(
    estimator=base_xgb,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1 # To see the progress
)

# 4. Fit GridSearchCV to the RFE-selected training data
print("Starting GridSearchCV fit...")
grid_search.fit(X_train_rfe, y_train.values.ravel())
print("GridSearchCV fit complete.")

# 5. Print the best parameters found by GridSearchCV
print("\nBest parameters found by GridSearchCV:")
print(grid_search.best_params_)

# 6. Print the best cross-validation score found by GridSearchCV
print("\nBest cross-validation accuracy score:")
print(f"{grid_search.best_score_:.4f}")

# 7. Get the best estimator (model) from GridSearchCV
best_xgb_model = grid_search.best_estimator_

# 8. Make predictions on the RFE-selected test data using the best estimator
y_pred_xgb_tuned = best_xgb_model.predict(X_test_rfe)

# 9. Print the classification report and accuracy score for the best model on the test set
print("\nClassification Report for Tuned XGBoost Model:")
print(classification_report(y_test, y_pred_xgb_tuned))

accuracy_xgb_tuned = accuracy_score(y_test, y_pred_xgb_tuned)
print(f"Accuracy Score for Tuned XGBoost Model: {accuracy_xgb_tuned:.4f}")

Reasoning: To implement RandomizedSearchCV, I will import the required classes and modules, initialize the XGBoost classifier, define the hyperparameter distributions, set up RandomizedSearchCV with the specified parameters, fit it to the training data, and finally evaluate the best model found.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import randint, uniform

# 1. Initialize XGBClassifier instance with base parameters
# The objective and num_class are fixed for this multiclass problem
# random_state for reproducibility
base_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train['Attack_type'].unique()),
    eval_metric='mlogloss',
    random_state=42
)

# 2. Define a parameter distribution for the XGBoost Classifier
param_distributions = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2)
}

# 3. Initialize RandomizedSearchCV
# n_iter=50 for 50 different combinations (can be adjusted)
# cv=3 for 3-fold cross-validation
# scoring='accuracy' to optimize for accuracy
# n_jobs=-1 to use all available CPU cores for parallel processing
random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_distributions,
    n_iter=50,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42 # For reproducibility of random sampling
)

# 4. Fit RandomizedSearchCV to the RFE-selected training data
print("Starting RandomizedSearchCV fit...")
random_search.fit(X_train_rfe, y_train.values.ravel())
print("RandomizedSearchCV fit complete.")

# 5. Print the best parameters found by RandomizedSearchCV
print("\nBest parameters found by RandomizedSearchCV:")
print(random_search.best_params_)

# 6. Print the best cross-validation score found by RandomizedSearchCV
print("\nBest cross-validation accuracy score:")
print(f"{random_search.best_score_:.4f}")

# 7. Get the best estimator (model) from RandomizedSearchCV
best_xgb_model_rs = random_search.best_estimator_

# 8. Make predictions on the RFE-selected test data using the best estimator
y_pred_xgb_tuned_rs = best_xgb_model_rs.predict(X_test_rfe)

# 9. Print the classification report and accuracy score for the best model on the test set
print("\nClassification Report for Tuned XGBoost Model (RandomizedSearchCV):")
print(classification_report(y_test, y_pred_xgb_tuned_rs))

accuracy_xgb_tuned_rs = accuracy_score(y_test, y_pred_xgb_tuned_rs)
print(f"Accuracy Score for Tuned XGBoost Model (RandomizedSearchCV): {accuracy_xgb_tuned_rs:.4f}")

I used two hyperparameter tuning methods, GridSearchCV and RandomizedSearchCV, to optimize the XGBoost Classifier model on the RFE-selected features. Both methods aimed to find the best combination of n_estimators, max_depth, and learning_rate.

**GridSearchCV Results**

•	Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 100}

•	Best Cross-Validation Accuracy Score: 0.9801

•	Test Set Accuracy: 0.9791

•	Classification Report: (Refer to previous output cell)

o	Showed high precision, recall, and f1-scores for most classes, with some minority classes (e.g., 6 and 9) still exhibiting lower performance (f1-scores around 0.48-0.56).

**RandomizedSearchCV Results**

•	Best Parameters: {'learning_rate': 0.08125956761539498, 'max_depth': 8, 'n_estimators': 178}

•	Best Cross-Validation Accuracy Score: 0.9800

•	Test Set Accuracy: 0.9791

•	Classification Report: (Refer to previous output cell)

o	Similar to GridSearchCV, it achieved strong performance for major classes and faced challenges with minority classes.

**Comparison:**

**Optimal Hyperparameters: **

Both approaches discovered similar regions for optimal parameters. GridSearchCV selected n_estimators=100, max_depth=7, and learning_rate=0.1, while RandomizedSearchCV—sampling more broadly—arrived at n_estimators=178, max_depth=8, and a learning_rate near 0.08. These minor differences imply either a delicate balance at the global optimum or a fairly flat performance surface around these values.

**Performance Comparison:**

The models tuned by both GridSearchCV and RandomizedSearchCV achieved almost identical accuracy scores (0.9791) on the test set. Their classification reports mirrored each other across all classes, indicating that, for this dataset and hyperparameter range, each method produced models with equally strong predictive abilities.

**Computational Trade-offs:**

GridSearchCV completed 54 training runs (18 parameter sets × 3 folds), systematically evaluating every combination within its grid—a thorough but potentially costly approach as the size of the parameter space grows.
RandomizedSearchCV ran 150 fits (50 randomly chosen combinations × 3 folds). By sampling from distributions rather than exhaustively testing fixed points, it covered more possible options efficiently, especially beneficial when dealing with numerous hyperparameters.

**Efficiency: **

In this case, RandomizedSearchCV was arguably more efficient, searching a broader parameter space through more iterations (50 vs. 18 combinations) and still returning performance equal to that of GridSearchCV. For larger search spaces or higher computational demands per fit, RandomizedSearchCV’s efficiency becomes even more apparent.

**Conclusion**

Both GridSearchCV and RandomizedSearchCV successfully optimized the XGBoost Classifier, yielding models that performed equally well on the test set. RandomizedSearchCV stands out as a flexible and practical choice for hyperparameter tuning, particularly when facing complex models or extensive parameter spaces, as it can pinpoint strong configurations without the need for exhaustive searches.



Let's use SHAP to understand the feature importance and individual predictions of our best_xgb_model.



In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd # Import pandas for Series
import seaborn as sns # Import seaborn for barplot

# Assuming 'best_xgb_model' is your trained XGBoost model from GridSearchCV or RandomizedSearchCV
# and X_test_rfe is your RFE-selected test data.

# Create a SHAP Tree Explainer for the XGBoost model
explainer = shap.TreeExplainer(best_xgb_model)

# Calculate SHAP values for the test set
# For multiclass, TreeExplainer typically returns a list of arrays, one for each class.
# Each array in the list has shape (num_samples, num_features)
shap_values_raw = explainer.shap_values(X_test_rfe)

# Ensure shap_values_raw is treated as a list of arrays for consistent processing
# If shap_values_raw is already a 3D array, convert it to a list of 2D arrays (one per class)
if isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    # If it's already (num_classes, num_samples, num_features), convert to list of (num_samples, num_features)
    shap_values_list = [shap_values_raw[i] for i in range(shap_values_raw.shape[0])]
elif isinstance(shap_values_raw, list):
    # Already a list of arrays
    shap_values_list = shap_values_raw
else:
    # This case should ideally not happen for multiclass XGBoost
    print("Warning: shap_values_raw is neither a list nor a 3D array. Assuming single output.")
    shap_values_list = [shap_values_raw] # Treat as a single class output

# Determine the actual number of features expected from X_test_rfe
num_features_in_X = X_test_rfe.shape[1]

# Check if SHAP values have more features than X_test_rfe
# If so, truncate the SHAP values to match X_test_rfe's feature count
# This assumes any extra features are at the end (e.g., bias term in some SHAP versions)
truncated_shap_values_list = []
for class_shap_values in shap_values_list:
    if class_shap_values.shape[1] > num_features_in_X:
        truncated_shap_values_list.append(class_shap_values[:, :num_features_in_X])
    else:
        truncated_shap_values_list.append(class_shap_values)

# Calculate global feature importance by averaging absolute SHAP values
# across all samples and all classes, for each feature.
# This results in a 1D array of shape (num_features,).
# Stack the list of 2D arrays into a 3D array: (num_classes, num_samples, num_features)
stacked_truncated_shap_values = np.stack(truncated_shap_values_list, axis=0)

# Calculate mean absolute SHAP value across classes (axis 0) and samples (axis 1)
global_shap_importances = np.mean(np.abs(stacked_truncated_shap_values), axis=(0, 1)) # Result is (num_features,)

# Create a Pandas Series for better display with feature names
global_shap_importances_series = pd.Series(global_shap_importances, index=X_test_rfe.columns)
global_shap_importances_series = global_shap_importances_series.sort_values(ascending=False)

# SHAP Bar Plot for Global Feature Importance
print("\nSHAP Bar Plot (Global Feature Importance, averaged across classes and samples):")
plt.figure(figsize=(12, 8))
sns.barplot(x=global_shap_importances_series.values, y=global_shap_importances_series.index, palette='viridis', hue=global_shap_importances_series.index, legend=False)
plt.title('Global Feature Importance from SHAP (Mean Absolute SHAP Value)')
plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# SHAP Force Plot (Local Explanation for a single prediction)
# Let's pick an arbitrary instance from the test set, e.g., the first one (index 0)
predicted_class_idx = best_xgb_model.predict(X_test_rfe.iloc[[0]])[0]
print(f"\nLocal Explanation for the first test instance (Predicted Class: {predicted_class_idx}):")

# For force_plot, use the truncated SHAP values for the specific class and sample
# to ensure feature count matches X_test_rfe.iloc[[0]]
force_plot_shap_values = truncated_shap_values_list[predicted_class_idx][0, :]

shap.force_plot(explainer.expected_value[predicted_class_idx], force_plot_shap_values, X_test_rfe.iloc[[0]], matplotlib=True)

Let's compare the global SHAP feature importances we just calculated with the built-in feature importances provided directly by the best_xgb_model.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Get Built-in Feature Importances from the best_xgb_model
# XGBoost's built-in feature importance is usually 'gain' or 'weight' by default.
# It's an array, so we need to map it back to feature names.
builtin_importances = best_xgb_model.feature_importances_

# Create a Pandas Series for built-in importances
builtin_importance_series = pd.Series(builtin_importances, index=X_test_rfe.columns)
builtin_importance_series_sorted = builtin_importance_series.sort_values(ascending=False)

print("Built-in Feature Importances from Tuned XGBoost Model (Sorted):")
display(builtin_importance_series_sorted)

# 2. Combine and Compare
# Ensure both series have the same features and are aligned
comparison_df = pd.DataFrame({
    'SHAP Importance': global_shap_importances_series,
    'Built-in Importance': builtin_importance_series_sorted
}).sort_values(by='SHAP Importance', ascending=False)

print("\nComparison DataFrame (Sorted by SHAP Importance):")
display(comparison_df)

# 3. Visualize the Comparison
plt.figure(figsize=(14, 8))
comparison_df.plot(kind='barh', figsize=(14, 8), colormap='coolwarm')
plt.title('Comparison of SHAP vs. Built-in Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.gca().invert_yaxis() # Display highest importance at the top
plt.tight_layout()
plt.show()